### Time-Independent One-Dimensional (1-D) Schrödinger Equation 
###### $ -\dfrac{\hbar^2}{2m} \, \dfrac{\mathrm{d}^2 \psi}{\mathrm{d} x^2} + V(x)\psi = H\psi = E\psi $ ( $ Ax = \lambda x $ )

### 2025 Spring "EE211: Physical Electronics"

###### Update history
###### 2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code. (v1)
###### 2024. 06. 26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
###### 2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization (plotly) and the flexibility of the shape of the potential. (v2)

###### ref
###### - Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p31
###### - John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)



# Time Independent Schrödinger Equation (TISE)

---

## Basic Formulation

- The TISE describes the stationary states of a quantum system and is given by:  
  $$
  \hat{H}\psi(x) = E\psi(x),
  $$  
  where:
  - $\hat{H}$ is the Hamiltonian operator.
  - $\psi(x)$ is the eigenfunction (wavefunction).
  - $E$ is the corresponding energy eigenvalue.

- For a single particle in one dimension, the Hamiltonian typically includes the kinetic and potential energy terms:  
  $$
  \hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + V(x).
  $$  
  Here, the kinetic term is represented by the second derivative with respect to $x$, and $V(x)$ is the potential energy function.

---

## Discretization on a Real Space Grid

- **Grid Representation:**  
  The continuous space is discretized into a grid with spacing $\Delta x$. The wavefunction $\psi(x)$ is then represented by its values on these grid points.

- **Finite Difference Approximation:**  
  The second derivative $\frac{d^2}{dx^2}$ in the kinetic term is approximated using finite difference methods. For example, a central difference scheme may be used:
  $$
  \frac{d^2\psi}{dx^2} \approx \frac{\psi(x+\Delta x) - 2\psi(x) + \psi(x-\Delta x)}{(\Delta x)^2}.
  $$

- **Hamiltonian Matrix Construction:**  
  Using the finite difference approximation, the Hamiltonian becomes a matrix where:
  - The diagonal elements typically contain $V(x)$ plus the central finite difference coefficient.
  - The off-diagonal elements represent the coupling between neighboring grid points due to the kinetic term.

---

## Eigenvalue Problem

- Once the Hamiltonian matrix is constructed, the TISE reduces to an eigenvalue problem:
  $$
  H \vec{\psi} = E \vec{\psi},
  $$  
  where $\vec{\psi}$ is the discretized wavefunction vector.

- **Numerical Solving:**  
  Standard numerical methods (e.g., using linear algebra libraries) can be used to solve for the eigenvalues $E$ and eigenvectors $\vec{\psi}$.  
  These eigenvalues represent the allowed energy levels of the system, while the eigenvectors provide the spatial distribution of the quantum states.

---

## Boundary Conditions

- **Dirichlet Boundary Conditions:**  
  For finite systems, the wavefunction is set to zero at the boundaries:
  $$
  \psi(x=0) = \psi(x=L) = 0.
  $$  
  This condition mimics an infinite potential outside the domain.



In [1]:
# install required libraries (run once)
import sys
print("Python executable:", sys.executable)

!{sys.executable} -m pip install -q --upgrade numpy scipy matplotlib ipython ipywidgets widgetsnbextension jupyterlab_widgets plotly nbformat tqdm

import ipywidgets as widgets
print("ipywidgets:", widgets.__version__)

import nbformat
print("nbformat:", nbformat.__version__)


Python executable: /home/class/anaconda3/bin/python


ipywidgets: 8.1.8
nbformat: 5.10.4


In [ ]:
import numpy as np
import numpy.linalg as lin
import math
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import HTML, display
import plotly.graph_objects as go
from scipy.constants import physical_constants

# Set numpy print options for debugging (optional)
np.set_printoptions(threshold=784, linewidth=np.inf)

###############################################################################
# Global Constants and Unit Conversions
###############################################################################
bohr_radius_ang = physical_constants['Bohr radius'][0] * 1e10  # in Angstrom
ang2bohr = 1 / bohr_radius_ang  # ~1.88973
har2ev = physical_constants['Hartree energy'][0] / physical_constants['electron volt'][0]  # ~27.21140795
laplacianDim = 3

###############################################################################
# Laplacian Functions (Finite Difference Method)
###############################################################################
def laplacianCoeff(laplacianDim): 
    a = np.zeros((2*laplacianDim+1, 2*laplacianDim+1))
    c = np.zeros(2*laplacianDim+1)
    c[2] = math.factorial(2)
    for i in range(2*laplacianDim+1):
        for j in range(2*laplacianDim+1):
            a[i,j] = (j - laplacianDim)**i
    inva = lin.inv(a)
    b = np.matmul(inva, c)
    lcoeff = b[laplacianDim:2*laplacianDim+1]
    return lcoeff

def laplacian(ngridx, laplacianDim, dx):
    lcoeff = laplacianCoeff(laplacianDim)
    loprt = np.zeros((ngridx, ngridx))
    for i in range(ngridx):
        for j in range(-laplacianDim, laplacianDim+1):
            k = i + j
            if k >= ngridx:
                k -= ngridx
            elif k < 0:
                k += ngridx
            loprt[i][k] = lcoeff[abs(j)] / (dx**2)
    return loprt

###############################################################################
# Potential and Hamiltonian Functions
###############################################################################
def potential(ngridx, pot_shape=0, pot_height_eV=25, curvature=0.5, width=0, slope=0.5):
    pot_height_har = pot_height_eV / har2ev
    pot_grid = np.zeros(ngridx)
    pot_grid[0] = 1e9
    pot_grid[ngridx-1] = 1e9

    if pot_shape == 0:      # Finite square well
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[int(ngridx * (0.5 - width)) : int(ngridx * (0.5 + width))] = 0
    elif pot_shape == 1:    # infinite square well
        pot_grid[1:ngridx] = 1e9
        pot_grid[int(ngridx * (0.5 - width)) : int(ngridx * (0.5 + width))] = 0        
    elif pot_shape == 2:    # Symmetric double square well
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[int(ngridx * (0.5 - width/2 - width)) : int(ngridx * (0.5 - width/2 ))] = 0
        pot_grid[int(ngridx * (0.5 + width/2 )) : int(ngridx * (0.5 + width/2 + width))] = 0
    elif pot_shape == 3:    # Harmonic
        for i in range(1, ngridx):
            pot_grid[i] = (i - (ngridx-1)//2)**2
        pot_grid[1:ngridx] = pot_grid[1:ngridx] * pot_height_har / (((ngridx-1)//2)**2) / curvature * 30
    elif pot_shape == 4:    # Triangular
        pot_grid[:] = 1e6
        for i in range(int(ngridx*0.4), ngridx-1):
            pot_grid[i] = (pot_height_har * abs(i - ngridx*0.4) / (ngridx*0.6)) * slope

        
    return pot_grid

def hamiltonian(ngridx, laplacianDim, dx, pot_height_eV=25, pot_shape=0, curvature=0.5, width=0, slope=0.6):
    pot_op = np.zeros((ngridx, ngridx))
    pot_1d = potential(ngridx, pot_shape, pot_height_eV, curvature, width, slope)
    laplacian_op = laplacian(ngridx, laplacianDim, dx)
    np.fill_diagonal(pot_op, val=pot_1d)
    ham = -laplacian_op/2. + pot_op
    return ham

def time_inde_schr_eq(pot_shape=0, curvature=0.5, width=0, slope=0.6, pot_height_eV=25):
    print("\n(1) Run Simulation...")
    ham_op = hamiltonian(ngridx, laplacianDim, dx, pot_height_eV, pot_shape, curvature, width, slope)
    (eigval, eigvec) = np.linalg.eigh(ham_op)
    print("\nDone!!")
    return eigval, eigvec


def compute_energy_components(eigval, eigvec, numOFstate, pot_shape, curvature, width, slope, pot_height_eV):
    # T_op = -laplacian/2, V_op = diag(potential)
    T_op = -laplacian(ngridx, laplacianDim, dx) / 2.
    V_1d = potential(ngridx, pot_shape, pot_height_eV, curvature, width, slope)
    V_op = np.diag(V_1d)

    pot = potential(ngridx, pot_shape, pot_height_eV, curvature, width, slope) * har2ev
    filtered_indices = [i for i in range(len(eigval)) if eigval[i] * har2ev < pot[650]]
    num_to_print = min(numOFstate, len(filtered_indices))
    print("\nEnergy components for filtered eigenstates (used in animation):")
    for idx in range(num_to_print):
        j = filtered_indices[idx]
        psi = eigvec[:, j]
        T_exp = np.vdot(psi, T_op @ psi).real
        V_exp = np.vdot(psi, V_op @ psi).real
        total = T_exp + V_exp
        print(f"State {j+1}: Kinetic = {T_exp * har2ev:.2f} eV, Potential = {V_exp * har2ev:.2f} eV, Total = {total * har2ev:.2f} eV")

# Animation Function: Animate the time evolution of multiple eigenstates' real, imaginary, and probability density
# Only animate eigenstates with eigenenergy below the potential height.
def animate_eigenstate(eigval, eigvec, numOFstate, boxL, ngridx, pot_shape, pot_height_eV, curvature, width, slope, y_max, sim_time):

    print("\n(2) Visualizing ...")
    print("\tRed \t= Real part of Ψ \n\tBlue \t= Imaginary part of Ψ \n\tGrey shaded = Probability Ψ*Ψ")
    # Define the x-axis in Angstrom units
    x = np.linspace(-boxL/2, boxL/2, ngridx) / ang2bohr
    pot = potential(ngridx, pot_shape, pot_height_eV, curvature, width, slope) * har2ev
    height = 2  # eV, coefficient for visualization

    # Filter eigenstates with eigenenergy below the potential height
    filtered_indices = [i for i in range(len(eigval)) if eigval[i] * har2ev < pot[650]]
    
    num_to_plot = min(numOFstate, len(filtered_indices))
    if num_to_plot < 1:
        print("\nNo eigenstate under the selected potential threshold.")
        return

    if num_to_plot == len(filtered_indices):
        print("\n*** Draw the eigenstates lower than the height of the potential well")
    
    # Compute normalization and energy offsets for each filtered eigenstate
    len_psi_list = [np.sum(np.abs(eigvec[:, j])**2) for j in filtered_indices[:num_to_plot]]
    E_eV_list = [eigval[j] * har2ev for j in filtered_indices[:num_to_plot]]
    E_J_list  = [eigval[j] * physical_constants["Hartree energy"][0] for j in filtered_indices[:num_to_plot]]

    
    
    hbar = physical_constants["reduced Planck constant"][0]
    sim_time_s = sim_time * 1e-15  # Convert fsec to seconds
    time_step = 1e-17             # 0.01 fsec
    t_values = np.arange(0, sim_time_s + time_step, time_step)

    def _lightness_by_energy_rank(idx, n, low=35.0, high=72.0):
        if n <= 1:
            return 0.5 * (low + high)
        ratio = idx / (n - 1)
        return low + (high - low) * ratio

    real_colors = [
        f"hsl(6, 88%, {_lightness_by_energy_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    imag_colors = [
        f"hsl(215, 86%, {_lightness_by_energy_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    fill_colors = []
    for i in range(num_to_plot):
        ratio = 0.0 if num_to_plot <= 1 else i / (num_to_plot - 1)
        alpha = 0.24 + 0.18 * ratio
        fill_colors.append(f"rgba(120,120,120,{alpha:.3f})")

    def _state_curves(state_idx, t):
        j = filtered_indices[state_idx]
        phase = np.exp(-1j * E_J_list[state_idx] * t / hbar)
        psi_t = eigvec[:, j] * phase
        real_scaled = psi_t.real / len_psi_list[state_idx] * height + E_eV_list[state_idx]
        imag_scaled = psi_t.imag / len_psi_list[state_idx] * height + E_eV_list[state_idx]
        prob_scaled = ((psi_t.real)**2 + (psi_t.imag)**2) / len_psi_list[state_idx] * height * 6 + E_eV_list[state_idx]
        baseline = np.full_like(x, E_eV_list[state_idx])
        x_fill = np.concatenate([x, x[::-1]])
        y_fill = np.concatenate([prob_scaled, baseline[::-1]])
        return real_scaled, imag_scaled, x_fill, y_fill

    static_annotations = []
    x_text = 8  # Angstrom
    for idx in range(num_to_plot):
        if (E_eV_list[idx] + y_max / 40.0) < y_max:
            static_annotations.append(
                dict(
                    x=x_text,
                    y=E_eV_list[idx] + y_max / 40.0,
                    text=f"{E_eV_list[idx]:.2f} eV",
                    showarrow=False,
                    font=dict(size=10, color="black"),
                    xanchor="left",
                    yanchor="middle",
                )
            )

    initial_data = [
        go.Scatter(
            x=x,
            y=pot,
            mode="lines",
            name="Potential",
            line=dict(color="black", width=1.2),
        )
    ]

    for idx in range(num_to_plot):
        real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t_values[0])
        initial_data.append(
            go.Scatter(
                x=x,
                y=real_scaled,
                mode="lines",
                name=f"Re(Ψ) state {idx + 1}",
                line=dict(color=real_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x,
                y=imag_scaled,
                mode="lines",
                name=f"Im(Ψ) state {idx + 1}",
                line=dict(color=imag_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x_fill,
                y=y_fill,
                mode="lines",
                name=f"|Ψ|² state {idx + 1}",
                line=dict(color="rgba(0,0,0,0)", width=0),
                fill="toself",
                fillcolor=fill_colors[idx],
            )
        )

    frames = []
    dynamic_trace_indices = list(range(1, 1 + 3 * num_to_plot))
    for iframe, t in enumerate(t_values):
        frame_data = []
        for idx in range(num_to_plot):
            real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t)
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=real_scaled,
                    mode="lines",
                    line=dict(color=real_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=imag_scaled,
                    mode="lines",
                    line=dict(color=imag_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x_fill,
                    y=y_fill,
                    mode="lines",
                    line=dict(color="rgba(0,0,0,0)", width=0),
                    fill="toself",
                    fillcolor=fill_colors[idx],
                )
            )

        frames.append(
            go.Frame(
                data=frame_data,
                traces=dynamic_trace_indices,
                name=str(iframe),
                layout=go.Layout(title_text=f"Time Evolution of Eigenstates (Time: {t * 1e15:.2f} fsec)"),
            )
        )

    slider_steps = []
    for iframe, t in enumerate(t_values):
        slider_steps.append(
            {
                "args": [[str(iframe)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
                "label": f"{t * 1e15:.2f}",
                "method": "animate",
            }
        )

    fig = go.Figure(data=initial_data, frames=frames)
    fig.update_layout(
        width=920,
        height=700,
        template="plotly_white",
        title=f"Time Evolution of Eigenstates (Time: {t_values[0] * 1e15:.2f} fsec)",
        margin=dict(l=80, r=220, t=90, b=140),
        xaxis=dict(title="Box [Angstrom]", range=[-15, 15]),
        yaxis=dict(title="Energy [eV]", range=[-0.125 * y_max, y_max]),
        annotations=static_annotations,
        updatemenus=[
            {
                "type": "buttons",
                "direction": "left",
                "showactive": True,
                "x": 0.0,
                "y": -0.06,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [None, {"fromcurrent": True, "frame": {"duration": 50, "redraw": True}, "transition": {"duration": 0}}],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}],
                    },
                ],
            }
        ],
        sliders=[
            {
                "active": 0,
                "x": 0.22,
                "y": -0.06,
                "len": 0.76,
                "currentvalue": {"prefix": "Time [fsec]: "},
                "pad": {"t": 40},
                "steps": slider_steps,
            }
        ],
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255,255,255,0.85)"
        ),
    )

    print("\n")
    print("Done!!")

    display(HTML(fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False)))

###############################################################################
# Interactive Widget
###############################################################################
if __name__=="__main__":
    boxL_ang = 100        # Box Length in Angstrom
    ngridx = 1001         # Number of grid points
    laplacianDim = 3
    boxL = boxL_ang * ang2bohr
    dx = boxL / ngridx

    widget_width = '620px'
    widget_description_width = '270px'
    widget_style = {'description_width': widget_description_width}

    def make_widget_layout(display=''):
        return widgets.Layout(width=widget_width, display=display, margin='12px 0px 12px 0px')

    def make_dropdown(options, value, description):
        return widgets.Dropdown(
            options=options,
            value=value,
            description=description,
            style=widget_style,
            layout=make_widget_layout(),
        )

    def make_float_slider(value, min, max, step, description, readout_format='.2f'):
        return widgets.FloatSlider(
            value=value,
            min=min,
            max=max,
            step=step,
            description=description,
            readout=True,
            readout_format=readout_format,
            continuous_update=False,
            style=widget_style,
            layout=make_widget_layout(),
        )

    def make_int_slider(value, min, max, step, description):
        return widgets.IntSlider(
            value=value,
            min=min,
            max=max,
            step=step,
            description=description,
            readout=True,
            continuous_update=False,
            style=widget_style,
            layout=make_widget_layout(),
        )

    pot_shape_widget = make_dropdown(
        options=[
            ('Finite Square Well', 0),
            ('Infinite Square Well', 1),
            ('Double Square Well', 2),
            ('Harmonic Oscillator', 3),
            ('Triangular Well', 4),
        ],
        value=0,
        description='Potential Shape:',
    )
    curvature_widget = make_float_slider(
        value=0.4, min=0.1, max=1, step=0.1,
        description='Curvature of harmonic well:',
        readout_format='.1f',
    )
    width_widget = make_float_slider(
        value=5, min=2, max=20, step=1,
        description='Width [Ang]:',
        readout_format='.1f',
    )
    slope_widget = make_float_slider(
        value=3, min=0.1, max=5, step=0.1,
        description='Slope of triangular well:',
        readout_format='.1f',
    )
    pot_height_widget = make_float_slider(
        value=4, min=0.5, max=10, step=0.1,
        description='Potential Height [eV]:',
        readout_format='.1f',
    )
    y_max_widget = make_float_slider(
        value=5, min=2, max=50, step=1,
        description='y range [eV]:',
        readout_format='.1f',
    )
    sim_time_widget = make_float_slider(
        value=0.1, min=0.1, max=10, step=0.1,
        description='Simulation Time [fsec]:',
        readout_format='.1f',
    )
    numOFstate_widget = make_int_slider(
        value=5, min=1, max=10, step=1,
        description='# of States:',
    )

    def set_widget_visible(widget, visible):
        widget.layout = make_widget_layout(display='' if visible else 'none')
    
    def update_visibility(change):
        value = pot_shape_widget.value
        set_widget_visible(curvature_widget, value == 3)
        set_widget_visible(width_widget, value <= 2)
        set_widget_visible(pot_height_widget, value in [0, 2])
        set_widget_visible(slope_widget, value == 4)

    set_widget_visible(curvature_widget, False)
    set_widget_visible(slope_widget, False)
    pot_shape_widget.observe(update_visibility, names='value')

    @interact_manual(
        numOFstate = numOFstate_widget,
        pot_shape  = pot_shape_widget,
        curvature = curvature_widget,
        width     = width_widget,
        slope     = slope_widget,
        pot_height_eV = pot_height_widget,
        y_max     = y_max_widget,
        sim_time  = sim_time_widget
    )
    def interactive(numOFstate, pot_shape, curvature, width, slope, pot_height_eV, y_max, sim_time):
        (eigval, eigvec) = time_inde_schr_eq(pot_shape, curvature, width/200, slope, pot_height_eV) # dividiing 200 = change it to Anstrom

        compute_energy_components(eigval, eigvec, numOFstate, pot_shape, curvature, width/200, slope, pot_height_eV)
        animate_eigenstate(eigval, eigvec, numOFstate, boxL, ngridx, pot_shape, pot_height_eV, curvature, width/200, slope, y_max, sim_time)

    readme = r"""
    """
    print(readme)
    update_visibility(None)
    pass


interactive(children=(IntSlider(value=5, continuous_update=False, description='# of States:', layout=Layout(di…